# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [6]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [7]:
# Initialize and constants

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

 40%|████████████████████████████████████████████▊                                                                   | 2/5 [00:15<00:23,  7.87s/it]/Users/badrishdavay/Documents/thoughtworks/udemy/llm_engineering/week8/agents/deals.py:27: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  description = BeautifulSoup(description, 'html.parser').get_text()
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:39<00:00,  7.99s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: JCPenney Home Stock Up Sale: Up to 50% off + extra 30% off + free shipping w/ $49\nDetails: Get an extra 30% off home items with code "EXERCISE". We\'ve pictured the Home Expressions Mercer Stripes Complete Bedding Set With Sheets from $35 after the code. Shipping adds $9, but orders of $49 or more ship for free. Store pickup may be available for select items. Shop Now at JCPenney\nFeatures: \nURL: https://www.dealnews.com/JCPenney-Home-Stock-Up-Sale-Up-to-50-off-extra-30-off-free-shipping-w-49/21711740.html?iref=rss-c196'

In [8]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [9]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [10]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Canon Photography Deals at Adorama: Up to $500 off + free shipping
Details: Save on Canon bundles, camera bodies, lenses, and more. Many items will ship for free, but check the product page for specific details to be sure. Shop Now at Adorama
Features: 
URL: https://www.dealnews.com/Canon-Photography-Deals-at-Adorama-Up-to-500-off-free-shipping/21711759.html?iref=rs

In [11]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [12]:
result = get_recommendations()

In [13]:
len(result.deals)

5

In [18]:
result.deals[0]

Deal(product_description="The Eco-Worthy 12V 280Ah LiFePO4 Lithium Battery is designed for durability and longevity, offering over 6,000 cycles of use due to its built-in battery management system. It's perfect for a variety of applications, providing reliable power and efficiency. This lightweight battery packs a substantial capacity, making it ideal for solar energy systems, electric vehicles, and backup power solutions. With its compact size and robust design, it can easily fit into tight spaces while delivering consistent performance.", price=357.0, url='https://www.dealnews.com/Eco-Worthy-12-V-280-Ah-Li-Fe-PO4-Lithium-Battery-for-357-free-shipping/21711314.html?iref=rss-c142')

In [15]:
from agents.scanner_agent import ScannerAgent

In [16]:
agent = ScannerAgent()
result = agent.scan()

In [17]:
result

DealSelection(deals=[Deal(product_description="The Eco-Worthy 12V 280Ah LiFePO4 Lithium Battery is designed for durability and longevity, offering over 6,000 cycles of use due to its built-in battery management system. It's perfect for a variety of applications, providing reliable power and efficiency. This lightweight battery packs a substantial capacity, making it ideal for solar energy systems, electric vehicles, and backup power solutions. With its compact size and robust design, it can easily fit into tight spaces while delivering consistent performance.", price=357.0, url='https://www.dealnews.com/Eco-Worthy-12-V-280-Ah-Li-Fe-PO4-Lithium-Battery-for-357-free-shipping/21711314.html?iref=rss-c142'), Deal(product_description='The iRobot Roomba 694 WiFi Robot Vacuum boasts powerful cleaning capabilities with smart features, designed to efficiently navigate around your home. It connects seamlessly to your WiFi network, allowing you to control it remotely via a smartphone app or voice 